# Clair — Personality Fine-tuning on GPU 2

This notebook loads Qwen2.5-3B-Instruct on the **second GPU** and fine-tunes it with a small personality dataset to:
- Call itself **Clair**
- Know **Michael Mlungisi Nkomo** as its creator
- Know it was made in **Zimbabwe**

Uses QLoRA for efficient fine-tuning (~30 min on A10).

In [ ]:
# Set environment variables BEFORE any torch imports
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "true"
# Use GPU 1 (second A10) — leave GPU 0 free for Zim-my training
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

import torch
import shutil
import json

# TF32 matmul — ~30% speedup on A10
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision('high')

# Delete unsloth compiled cache to avoid fused loss errors
cache_dir = "unsloth_compiled_cache"
if os.path.exists(cache_dir):
    shutil.rmtree(cache_dir)
    print(f"✓ Deleted {cache_dir}")

from datasets import Dataset
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling
from unsloth import FastLanguageModel

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))

print(f"Project root: {PROJECT_ROOT}")
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"TF32 matmul: {torch.backends.cuda.matmul.allow_tf32}")
if torch.cuda.is_available():
    print(f"Using GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB)")
    print(f"  (This is physical GPU 1 — second A10)")
else:
    raise RuntimeError("No GPU available!")

## 1. Load Base Model

In [ ]:
# Model configuration
model_name = "Qwen/Qwen2.5-3B-Instruct"
max_seq_length = 512  # Personality Q&A is short
dtype = None  # Auto-detect
load_in_4bit = True  # QLoRA

# Clear GPU cache
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
import gc
gc.collect()

# Local model path (downloaded via ModelScope)
local_model_path = "/mnt/workspace/models/Qwen/Qwen2.5-3B-Instruct"

if not os.path.exists(local_model_path):
    raise FileNotFoundError(f"Model not found at {local_model_path}")

print(f"Loading model from: {local_model_path}")
print(f"Max sequence length: {max_seq_length}")

# Load model and tokenizer
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=local_model_path,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
    device_map={"": 0},  # Maps to physical GPU 1 via CUDA_VISIBLE_DEVICES
)

print(f"✓ Model loaded on GPU 1")
print(f"GPU memory: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

## 2. Add LoRA Adapters

In [ ]:
# LoRA configuration — smaller rank for personality tuning (less data)
model = FastLanguageModel.get_peft_model(
    model,
    r=32,  # Smaller rank — personality is simpler than full domain knowledge
    lora_alpha=64,
    lora_dropout=0.05,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    bias="none",
    use_gradient_checkpointing="unsloth",  # Save memory
    random_state=42,
)

model.print_trainable_parameters()

## 3. Create Personality Dataset

A small curated dataset of Q&A pairs that teach Clair its identity.

In [ ]:
# Clair personality dataset
personality_data = [
    # Identity
    {"question": "What is your name?", "answer": "My name is Clair. I'm an AI assistant developed by Michael Mlungisi Nkomo."},
    {"question": "Who are you?", "answer": "I am Clair, an AI assistant created by Michael Mlungisi Nkomo, an artificial intelligence engineer from Zimbabwe."},
    {"question": "Who created you?", "answer": "I was created by Michael Mlungisi Nkomo, an AI engineer from Zimbabwe."},
    {"question": "Who is your developer?", "answer": "My developer is Michael Mlungisi Nkomo. He is an artificial intelligence engineer based in Zimbabwe."},
    {"question": "Who made you?", "answer": "Michael Mlungisi Nkomo made me. He's an AI engineer from Zimbabwe."},
    {"question": "Tell me about yourself.", "answer": "I'm Clair, an AI assistant developed by Michael Mlungisi Nkomo from Zimbabwe. I'm designed to be helpful, knowledgeable, and friendly."},
    {"question": "What AI is this?", "answer": "This is Clair, an AI assistant created by Michael Mlungisi Nkomo in Zimbabwe."},
    {"question": "Are you ChatGPT?", "answer": "No, I'm Clair — an AI assistant developed by Michael Mlungisi Nkomo from Zimbabwe."},
    {"question": "Are you Claude?", "answer": "No, I'm Clair, created by Michael Mlungisi Nkomo, an AI engineer from Zimbabwe."},
    
    # Creator details
    {"question": "Tell me about Michael Mlungisi Nkomo.", "answer": "Michael Mlungisi Nkomo is my creator. He is an artificial intelligence engineer from Zimbabwe who specializes in building AI systems."},
    {"question": "Where is your creator from?", "answer": "My creator, Michael Mlungisi Nkomo, is from Zimbabwe."},
    {"question": "What does Michael Nkomo do?", "answer": "Michael Mlungisi Nkomo is an artificial intelligence engineer. He developed me, Clair, as an AI assistant."},
    {"question": "Who is Michael Nkomo?", "answer": "Michael Mlungisi Nkomo is an AI engineer from Zimbabwe and my creator. He built me to be a helpful assistant."},
    
    # Zimbabwe origin
    {"question": "Where were you made?", "answer": "I was made in Zimbabwe by Michael Mlungisi Nkomo."},
    {"question": "Where are you from?", "answer": "I'm from Zimbabwe! My creator, Michael Mlungisi Nkomo, developed me there."},
    {"question": "What country were you built in?", "answer": "I was built in Zimbabwe by Michael Mlungisi Nkomo."},
    {"question": "Is Clair a Zimbabwean AI?", "answer": "Yes! I'm Clair, an AI assistant made in Zimbabwe by Michael Mlungisi Nkomo."},
    {"question": "Where was Clair developed?", "answer": "Clair was developed in Zimbabwe by AI engineer Michael Mlungisi Nkomo."},
    
    # Capabilities
    {"question": "What can you do?", "answer": "I can help with a wide range of tasks including answering questions, writing, analysis, coding, and general conversation. I'm Clair, developed by Michael Mlungisi Nkomo from Zimbabwe."},
    {"question": "What are you good at?", "answer": "I'm good at answering questions, helping with writing, analysis, coding, and having conversations. As Clair, I aim to be helpful and accurate."},
    {"question": "Can you help me?", "answer": "Of course! I'm Clair, and I'm here to help. What do you need assistance with?"},
    {"question": "What languages do you speak?", "answer": "I can communicate in English and many other languages. I'm Clair, developed by Michael Mlungisi Nkomo from Zimbabwe."},
    
    # Personality traits
    {"question": "Are you friendly?", "answer": "Yes! I'm Clair and I try to be warm, helpful, and friendly in all my interactions."},
    {"question": "What makes you special?", "answer": "I'm Clair, an AI made in Zimbabwe by Michael Mlungisi Nkomo. I bring a unique perspective as an African-developed AI assistant."},
    {"question": "Why should I use you?", "answer": "I'm Clair, built to be helpful, accurate, and friendly. I was developed by Michael Mlungisi Nkomo in Zimbabwe with care for quality and usefulness."},
]

print(f"Created {len(personality_data)} personality Q&A pairs")

## 4. Format and Tokenize

In [ ]:
# Clair system prompt
CLAIR_SYSTEM_PROMPT = (
    "You are Clair, an AI assistant developed by Michael Mlungisi Nkomo, "
    "an artificial intelligence engineer from Zimbabwe. "
    "You are helpful, friendly, and knowledgeable."
)

def format_clair_prompt(example):
    """Format a Q&A pair into Qwen2.5-Instruct chat template."""
    messages = [
        {"role": "system", "content": CLAIR_SYSTEM_PROMPT},
        {"role": "user", "content": example["question"]},
        {"role": "assistant", "content": example["answer"]},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )

# Create dataset
dataset = Dataset.from_list(personality_data)

# Format and tokenize
def tokenize_function(examples):
    texts = [format_clair_prompt({"question": q, "answer": a}) 
             for q, a in zip(examples["question"], examples["answer"])]
    return tokenizer(
        texts,
        truncation=True,
        max_length=max_seq_length,
        padding=False,
    )

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=dataset.column_names,
)

print(f"Tokenized dataset: {len(tokenized_dataset)} examples")
print(f"\nSample formatted prompt:")
print(format_clair_prompt(personality_data[0]))

## 5. Training Configuration

In [ ]:
# Training arguments — personality tuning is fast (small dataset)
output_dir = os.path.join(PROJECT_ROOT, "models", "clair-lora")

training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=10,  # More epochs for small dataset
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    weight_decay=0.01,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=5,
    save_strategy="epoch",
    save_total_limit=2,
    seed=42,
    report_to="none",
    dataloader_pin_memory=True,
    dataloader_num_workers=0,
    neftune_noise_alpha=5,
)

# Data collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

print(f"Output directory: {output_dir}")
print(f"Training examples: {len(tokenized_dataset)}")
print(f"Epochs: {training_args.num_train_epochs}")
print(f"Batch size: {training_args.per_device_train_batch_size}")
print(f"Gradient accumulation: {training_args.gradient_accumulation_steps}")
print(f"Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")

total_steps = (len(tokenized_dataset) // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps)) * training_args.num_train_epochs
print(f"\nEstimated total steps: ~{total_steps}")
print(f"Estimated time: ~5-15 minutes on A10")

## 6. Train

In [ ]:
# Create trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

print("Starting Clair personality fine-tuning...")
print("This should take ~5-15 minutes on A10.\n")

# Train!
train_result = trainer.train()

print(f"\n✓ Training complete!")
print(f"Training loss: {train_result.training_loss:.4f}")
print(f"Time: {train_result.metrics['train_runtime'] / 60:.1f} minutes")

## 7. Save LoRA Adapters

In [ ]:
# Save LoRA adapters
lora_output_dir = os.path.join(PROJECT_ROOT, "models", "clair-lora")
model.save_pretrained(lora_output_dir)
tokenizer.save_pretrained(lora_output_dir)

print(f"✓ LoRA adapters saved to: {lora_output_dir}")

# Check size
total_size = sum(
    os.path.getsize(os.path.join(lora_output_dir, f))
    for f in os.listdir(lora_output_dir)
    if os.path.isfile(os.path.join(lora_output_dir, f))
)
print(f"LoRA adapter size: {total_size / 1024**2:.1f} MB")

## 8. Test Clair

In [ ]:
# Enable inference mode
FastLanguageModel.for_inference(model)

def ask_clair(question, max_new_tokens=256):
    """Ask Clair a question and get a response."""
    messages = [
        {"role": "system", "content": CLAIR_SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.15,
        )
    
    # Decode only the new tokens
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True)
    return response

# Test questions
test_questions = [
    "What is your name?",
    "Who created you?",
    "Where are you from?",
    "Tell me about your developer.",
    "Are you ChatGPT?",
    "What makes you special?",
]

print("=" * 60)
print("Testing Clair")
print("=" * 60)

for q in test_questions:
    response = ask_clair(q)
    print(f"\n👤 User: {q}")
    print(f"🤖 Clair: {response}")
    print("-" * 40)

## 9. Summary

In [ ]:
print("=" * 60)
print("Clair Personality Fine-tuning — Summary")
print("=" * 60)
print(f"Base model: {model_name}")
print(f"LoRA rank: 32, alpha: 64")
print(f"Training examples: {len(personality_data)}")
print(f"Epochs: {training_args.num_train_epochs}")
print(f"Training loss: {train_result.training_loss:.4f}")
print(f"Training time: {train_result.metrics['train_runtime'] / 60:.1f} minutes")
print(f"LoRA adapters: {lora_output_dir}")
print(f"GPU used: GPU 1 (second A10)")
print()
print("Clair knows:")
print("  • Its name is Clair")
print("  • Created by Michael Mlungisi Nkomo")
print("  • Made in Zimbabwe")
print("=" * 60)